# 03 - Models (Deliverable 2)
Evaluates the saved cuisine classifiers on the **held-out test set** and the Word2Vec substitution model. Models were trained by `src/train_models.py` (70/15/15 stratified split, seed 42). The human rating study was skipped by choice; substitution is evaluated qualitatively + with the automatic hide-one-ingredient test.

In [1]:
import sys; sys.path.insert(0, '../src')
import joblib, numpy as np, pandas as pd, plotly.express as px
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from train_models import load_split
from pathlib import Path
REP = Path('../reports'); REP.mkdir(exist_ok=True)
rec, le, tr, va, te = load_split()
print(len(tr), len(va), len(te))
Xva, yva, Xte, yte = rec.loc[va,'doc'], rec.loc[va,'y'], rec.loc[te,'doc'], rec.loc[te,'y']

27826 5963 5963


## 1. Cuisine classifier

In [2]:
names = {'tfidf_logreg':'TF-IDF + LogReg (baseline)', 'linear_svc':'TF-IDF + LinearSVC', 'xgboost':'TF-IDF + XGBoost'}
models = {k: joblib.load(f'../models/{k}.joblib') for k in names}
rows, preds = [], {}
for k, m in models.items():
    for split, X, y in [('val', Xva, yva), ('test', Xte, yte)]:
        p = m.predict(X)
        rows.append({'model': names[k], 'split': split, 'accuracy': accuracy_score(y, p), 'macro_F1': f1_score(y, p, average='macro')})
        if split == 'test': preds[k] = p
res = pd.DataFrame(rows).pivot(index='model', columns='split', values=['accuracy','macro_F1']).round(4)
res.to_csv('../reports/classifier_results.csv'); res

accuracy         macro_F1        
split                          test     val     test     val
model                                                       
TF-IDF + LinearSVC           0.7822  0.7907   0.7009  0.7106
TF-IDF + LogReg (baseline)   0.7744  0.7803   0.6927  0.6999
TF-IDF + XGBoost             0.7582  0.7622   0.6755  0.6759

In [3]:
best = 'linear_svc'   # picked on VALIDATION scores above; test is only reported
cm = confusion_matrix(yte, preds[best], normalize='true')
fig = px.imshow(cm.round(2), x=le.classes_, y=le.classes_, color_continuous_scale='Blues', height=750,
                title=f'Confusion matrix (row-normalised) - {names[best]} - test set', labels=dict(x='predicted', y='true'))
fig.write_html(REP/'confusion_matrix.html'); fig.show()
print(classification_report(yte, preds[best], target_names=le.classes_, digits=3))

              precision    recall  f1-score   support

   brazilian      0.736     0.557     0.634        70
     british      0.650     0.430     0.517       121
cajun_creole      0.828     0.746     0.785       232
     chinese      0.762     0.855     0.806       401
    filipino      0.736     0.593     0.657       113
      french      0.596     0.612     0.604       397
       greek      0.732     0.699     0.715       176
      indian      0.836     0.906     0.870       449
       irish      0.609     0.390     0.476       100
     italian      0.816     0.889     0.851      1175
    jamaican      0.757     0.671     0.711        79
    japanese      0.861     0.700     0.772       213
      korean      0.860     0.736     0.793       125
     mexican      0.886     0.912     0.899       966
    moroccan      0.800     0.748     0.773       123
     russian      0.625     0.411     0.496        73
 southern_us      0.716     0.812     0.761       648
     spanish      0.583    

In [4]:
# most confused pairs (off-diagonal, row-normalised)
cmn = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
conf = cmn.stack(); conf = conf[[a!=b for a,b in conf.index]].sort_values(ascending=False).head(8)
print(conf.round(3).to_string())

vietnamese  thai           0.218
spanish     italian        0.203
greek       italian        0.199
french      italian        0.194
british     southern_us    0.182
irish       french         0.180
            southern_us    0.180
russian     french         0.164


## 2. Ingredient substitution (Word2Vec)

In [5]:
from substitute import Substituter
sub = Substituter()
print('vocab:', len(sub.w2v.wv))
queries = ['butter','chicken','soy sauce','basil','milk','shrimp','cheddar cheese','lemon juice','cumin','honey']
for q in queries:
    plain = [r[0] for r in sub.suggest(q, k=5, same_category=False, rerank=False)]
    full  = [f'{r[0]} ({r[1]:.2f})' for r in sub.suggest(q, k=5)]
    print(f'{q:15s} plain w2v: {plain}\n{"":15s} +category +rerank: {full}')

vocab: 2395
butter          plain w2v: ['milk', 'all purpose flour', 'heavy cream', 'heavy whipping cream', 'egg']
                +category +rerank: ['milk (0.48)', 'sour cream (0.38)', 'heavy cream (0.32)', 'heavy whipping cream (0.32)', 'cream cheese (0.30)']
chicken         plain w2v: ['beef', 'meat', 'pork butt', 'pork tenderloin', 'turkey']
                +category +rerank: ['pork butt (0.47)', 'beef (0.46)', 'pork tenderloin (0.46)', 'pork shoulder (0.45)', 'pork sausage (0.43)']
soy sauce       plain w2v: ['sesame oil', 'rice wine', 'rice vinegar', 'oyster sauce', 'snow pea']
                +category +rerank: ['sesame oil (0.26)', 'rice vinegar (0.21)', 'black vinegar (0.16)', 'chinese black vinegar (0.11)']
basil           plain w2v: ['italian turkey sausage', 'parmesan cheese', 'red bell pepper', 'olive oil', 'mozzarella cheese']
                +category +rerank: ['rosemary sprig (0.39)', 'rosemary (0.27)', 'parsley (0.20)']
milk            plain w2v: ['butter', 'heavy whi

### Automatic evaluation: hide one ingredient, recover it from the rest
On recipes from the **test split** (Word2Vec saw only train recipes). Context = mean vector of the remaining ingredients; hit@k = hidden ingredient in top-k.

In [6]:
w2v_vocab = set(sub.w2v.wv.key_to_index)
test_recipes = [r for r in rec.loc[te,'ings']]
r = sub.hide_one_test(test_recipes, n=3000)
# random baseline: expected hit@k = k / (vocab size - |context|)
V = len(w2v_vocab)
base = {f'hit@{k}': k / V for k in (1,5,10)}
print('model   :', {k: round(v,4) for k,v in r.items()})
print('random  :', {k: round(v,4) for k,v in base.items()})

model   : {'hit@1': 0.0263, 'hit@5': 0.1007, 'hit@10': 0.1487, 'MRR': 0.066, 'n': 3000}
random  : {'hit@1': 0.0004, 'hit@5': 0.0021, 'hit@10': 0.0042}


In [7]:
# Does category filtering / re-ranking change suggestions? share of top-5 neighbours in the same flavor category as the query
import itertools
vocab_q = [i for i in sub.w2v.wv.index_to_key[:300] if sub.cat.get(i)]
def same_cat_rate(**kw):
    tot = same = 0
    for q in vocab_q:
        for name, *_ in sub.suggest(q, k=5, **kw):
            c = sub.cat.get(name)
            if c is not None:
                tot += 1; same += (c == sub.cat[q])
    return same / tot
print('same-category rate, plain w2v      :', round(same_cat_rate(same_category=False, rerank=False), 3))
print('same-category rate, with filter    :', round(same_cat_rate(same_category=True, rerank=False), 3))
print('same-category rate, filter + rerank:', round(same_cat_rate(same_category=True, rerank=True), 3))

same-category rate, plain w2v      : 0.458


same-category rate, with filter    : 0.853


same-category rate, filter + rerank: 0.854


## 3. Findings (Deliverable 2)

**Cuisine classifier (held-out test set, 5,963 recipes, stratified 70/15/15 split).**

| model | test accuracy | test macro-F1 |
|---|---|---|
| TF-IDF + LogReg (baseline) | 0.774 | 0.693 |
| **TF-IDF + LinearSVC** (best on validation) | **0.782** | **0.701** |
| TF-IDF + XGBoost | 0.758 | 0.676 |

- The linear models beat gradient boosting here and train in seconds (XGBoost took ~7 minutes). With sparse bag-of-ingredients features, linear models are hard to beat.
- Macro-F1 (0.70) is well below accuracy (0.78) because of class imbalance: small classes have low recall - Irish 0.39, Russian 0.41, British 0.43, Spanish 0.43 - while Mexican (F1 0.90), Indian (0.87), Italian (0.85) are easy.
- Most confused pairs (share of true class predicted as other): Vietnamese->Thai 22%, Spanish->Italian 20%, Greek->Italian 20%, French->Italian 19%, British->Southern US 18%, Irish->French / Southern US 18%. These mirror the cuisine-similarity results from notebook 02 and are culinarily sensible.
- Not done (optional, out of scope): neural net, class weighting / resampling for the minority classes (the obvious next step to raise macro-F1).

**Substitution (Word2Vec, 100-d, skip-gram, each shuffled recipe a sentence, trained on train split only).**
- Plain nearest neighbours mix *substitutes* with *companions* (basil -> parmesan, mozzarella; soy sauce -> snow pea). Three fixes were added in `src/substitute.py`: (1) same flavor-category filter, (2) re-ranking = 0.7 x cosine + 0.3 x compound Jaccard - 0.25 x PMI with the query (penalises ingredients that usually appear *with* it), (3) drop name variants ("soy sauce" -> "light soy sauce") and rare ingredients (< 40 recipes).
- Share of top-5 suggestions in the same flavor category: 0.46 (plain) -> 0.85 (filtered). Good examples: butter -> milk, sour cream, heavy cream; honey -> sugar, maple syrup; shrimp -> prawn, clam; chicken -> pork, beef; basil -> rosemary, parsley.
- Weak spots: some category labels from the flavor data are coarse or odd (lemon juice -> golden raisin, honey -> granny smith apple); cheddar cheese -> "mexican cheese blend" is a context association rather than a true swap.
- **Automatic hide-one test** (test-split recipes, 3,000 trials): the model recovers the hidden ingredient in the top-1 / 5 / 10 in 2.6% / 10.1% / 14.9% of cases (MRR 0.066) vs 0.04% / 0.21% / 0.42% for random - roughly 35-65x better than chance, but this task is hard (many valid ingredients fit a recipe) so absolute numbers are low. Note this measures *recipe completion*, not substitution quality.
- **Human rating study was skipped (by user's decision)**, so substitution quality is supported only by the examples above and the automatic checks. A template for running the study later is easy to add (20 queries x top-3, ratings 1-5).